# 07 OpenOximetry Sample-Anchored Reference Join (Phase 3)

Join bloodgas and pulseox rows at encounter/sample granularity, then anchor pilot windows using 2Hz sample markers for Phase 2 eligible encounters.

Inputs from: `outputs/openox_ppg_eligible_encounters.csv` (from notebook 06), plus bloodgas and pulseox source tables.
Outputs to: `outputs/openox_sample_reference_pairs.csv` and `outputs/openox_sample_window_manifest.csv`.
Next: notebook 08 for raw PPG metadata and concrete sample-slice indices.

In [ ]:
from pathlib import Path

import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [start, *start.parents]:
    if (candidate / "data" / "physionet.org" / "files" / "openox-repo" / "1.1.1").is_dir() and (candidate / "notebooks").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find project root containing OpenOximetry data and notebooks/")

OPENOX_DIR = PROJECT_ROOT / "data" / "physionet.org" / "files" / "openox-repo" / "1.1.1"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

BLOODGAS_CSV = OPENOX_DIR / "bloodgas.csv"
PULSEOX_CSV = OPENOX_DIR / "pulseoximeter.csv"
ELIGIBLE_CSV = OUTPUTS_DIR / "openox_ppg_eligible_encounters.csv"

OPENOX_SAMPLE_REFERENCE_PAIRS_CSV = OUTPUTS_DIR / "openox_sample_reference_pairs.csv"
OPENOX_SAMPLE_WINDOW_MANIFEST_CSV = OUTPUTS_DIR / "openox_sample_window_manifest.csv"

for path in [BLOODGAS_CSV, PULSEOX_CSV, ELIGIBLE_CSV]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required Phase 3 input: {path}")

WINDOW_RADIUS_SECONDS = 15

print(f"Project root: {PROJECT_ROOT}")
print(f"OpenOximetry dir: {OPENOX_DIR}")
print(f"Eligible encounters file: {ELIGIBLE_CSV}")
print(f"Outputs dir: {OUTPUTS_DIR}")

## Build Encounter/Sample Reference Pairs

Filter to eligible encounters, aggregate bloodgas and pulseox by sample index, and build a paired reference table with encounter/sample-level coverage diagnostics.

In [ ]:
eligible_df = pd.read_csv(ELIGIBLE_CSV)
eligible_encounters = set(eligible_df["encounter_id"].astype(str))
eligible_path_map = eligible_df[["encounter_id", "ppg_hea_path", "hz2_csv_path"]].copy()

bloodgas_df = pd.read_csv(BLOODGAS_CSV, low_memory=False)
bloodgas_df = bloodgas_df.loc[bloodgas_df["encounter_id"].astype(str).isin(eligible_encounters)].copy()
bloodgas_df["encounter_id"] = bloodgas_df["encounter_id"].astype(str)
bloodgas_df["sample_num"] = pd.to_numeric(bloodgas_df.get("sample"), errors="coerce")
bloodgas_df["sample_num"] = bloodgas_df["sample_num"].round().astype("Int64")
bloodgas_df["so2"] = pd.to_numeric(bloodgas_df.get("so2"), errors="coerce")
bloodgas_df["bloodgas_timestamp"] = pd.to_datetime(
    bloodgas_df.get("date", "").astype(str).str.strip() + " " + bloodgas_df.get("time", "").astype(str).str.strip(),
    errors="coerce",
)
bloodgas_df = bloodgas_df.loc[bloodgas_df["sample_num"].notna() & bloodgas_df["so2"].notna()].copy()

bloodgas_grouped = (
    bloodgas_df.sort_values(["encounter_id", "sample_num", "bloodgas_timestamp"]).groupby(["encounter_id", "sample_num"], as_index=False).agg(
        bloodgas_row_count=("so2", "size"),
        so2_mean=("so2", "mean"),
        so2_min=("so2", "min"),
        so2_max=("so2", "max"),
        bloodgas_timestamp_min=("bloodgas_timestamp", "min"),
        bloodgas_timestamp_max=("bloodgas_timestamp", "max"),
    )
)

pulseox_df = pd.read_csv(PULSEOX_CSV, low_memory=False)
pulseox_df = pulseox_df.loc[pulseox_df["encounter_id"].astype(str).isin(eligible_encounters)].copy()
pulseox_df["encounter_id"] = pulseox_df["encounter_id"].astype(str)
pulseox_df["sample_num"] = pd.to_numeric(pulseox_df.get("sample_number"), errors="coerce").round().astype("Int64")
pulseox_df["saturation"] = pd.to_numeric(pulseox_df.get("saturation"), errors="coerce")
pulseox_df["pi"] = pd.to_numeric(pulseox_df.get("pi"), errors="coerce")
pulseox_df = pulseox_df.loc[pulseox_df["sample_num"].notna()].copy()

pulseox_grouped = (
    pulseox_df.groupby(["encounter_id", "sample_num"], as_index=False).agg(
        pulseox_row_count=("sample_num", "size"),
        pulseox_device_count=("device", "nunique"),
        pulseox_saturation_mean=("saturation", "mean"),
        pulseox_saturation_min=("saturation", "min"),
        pulseox_saturation_max=("saturation", "max"),
        pulseox_pi_mean=("pi", "mean"),
    )
)

reference_pairs_df = bloodgas_grouped.merge(
    pulseox_grouped,
    on=["encounter_id", "sample_num"],
    how="left",
)
reference_pairs_df["has_pulseox_match"] = reference_pairs_df["pulseox_row_count"].fillna(0).astype(int) > 0

print(f"Eligible encounters loaded: {len(eligible_encounters)}")
print(f"Encounter/sample bloodgas pairs: {len(reference_pairs_df)}")
print(f"Pairs with pulseox sample match: {int(reference_pairs_df['has_pulseox_match'].sum())}")

## Anchor To 2Hz Sample Markers And Create Window Manifest

Map each encounter/sample pair to the earliest available 2Hz anchor timestamp, derive fixed-duration windows around anchors, and export the canonical pilot manifest for Phase 4 indexing.

In [ ]:
def parse_2hz_sample_markers(hz2_csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(hz2_csv_path, usecols=["Sample", "Timestamp"], low_memory=False)
    df["sample_num"] = pd.to_numeric(df["Sample"], errors="coerce").round().astype("Int64")
    df["timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df = df.loc[df["sample_num"].notna() & df["timestamp"].notna(), ["sample_num", "timestamp"]].copy()
    if df.empty:
        return pd.DataFrame(columns=["sample_num", "anchor_timestamp", "sample_marker_count"])

    grouped = (
        df.groupby("sample_num", as_index=False).agg(
            anchor_timestamp=("timestamp", "min"),
            sample_marker_count=("timestamp", "size"),
        )
    )
    return grouped


anchor_rows = []
for row in eligible_path_map.itertuples(index=False):
    encounter_id = str(row.encounter_id)
    hz2_path = Path(row.hz2_csv_path)
    encounter_pairs = reference_pairs_df.loc[reference_pairs_df["encounter_id"] == encounter_id].copy()
    if encounter_pairs.empty:
        continue

    if not hz2_path.is_file():
        encounter_pairs["anchor_timestamp"] = pd.NaT
        encounter_pairs["sample_marker_count"] = 0
        anchor_rows.append(encounter_pairs)
        continue

    marker_df = parse_2hz_sample_markers(hz2_path)
    merged = encounter_pairs.merge(marker_df, on="sample_num", how="left")
    merged["sample_marker_count"] = merged["sample_marker_count"].fillna(0).astype(int)
    anchor_rows.append(merged)

if anchor_rows:
    anchored_pairs_df = pd.concat(anchor_rows, ignore_index=True)
else:
    anchored_pairs_df = reference_pairs_df.copy()
    anchored_pairs_df["anchor_timestamp"] = pd.NaT
    anchored_pairs_df["sample_marker_count"] = 0

anchored_pairs_df = anchored_pairs_df.merge(eligible_path_map, on="encounter_id", how="left")
anchored_pairs_df["has_2hz_sample_match"] = anchored_pairs_df["anchor_timestamp"].notna()
anchored_pairs_df["binary_label_so2_lt_90"] = anchored_pairs_df["so2_mean"] < 90.0

anchored_pairs_df = anchored_pairs_df.sort_values(["encounter_id", "sample_num"]).reset_index(drop=True)
anchored_pairs_df.to_csv(OPENOX_SAMPLE_REFERENCE_PAIRS_CSV, index=False)

window_manifest_df = anchored_pairs_df.loc[anchored_pairs_df["has_2hz_sample_match"]].copy()
window_manifest_df["window_center_timestamp"] = window_manifest_df["anchor_timestamp"]
window_manifest_df["window_start_timestamp"] = window_manifest_df["anchor_timestamp"] - pd.to_timedelta(WINDOW_RADIUS_SECONDS, unit="s")
window_manifest_df["window_end_timestamp"] = window_manifest_df["anchor_timestamp"] + pd.to_timedelta(WINDOW_RADIUS_SECONDS, unit="s")
window_manifest_df["window_duration_seconds"] = float(WINDOW_RADIUS_SECONDS * 2)
window_manifest_df["window_rule"] = f"sample-anchored +/- {WINDOW_RADIUS_SECONDS}s"

manifest_columns = [
    "encounter_id",
    "sample_num",
    "anchor_timestamp",
    "window_center_timestamp",
    "window_start_timestamp",
    "window_end_timestamp",
    "window_duration_seconds",
    "window_rule",
    "so2_mean",
    "so2_min",
    "so2_max",
    "binary_label_so2_lt_90",
    "bloodgas_row_count",
    "has_pulseox_match",
    "pulseox_row_count",
    "pulseox_device_count",
    "pulseox_saturation_mean",
    "sample_marker_count",
    "ppg_hea_path",
    "hz2_csv_path",
]
window_manifest_df = window_manifest_df[manifest_columns].sort_values(["encounter_id", "sample_num"]).reset_index(drop=True)
window_manifest_df.to_csv(OPENOX_SAMPLE_WINDOW_MANIFEST_CSV, index=False)

summary_df = pd.DataFrame(
    [
        {"metric": "eligible_encounters", "value": int(len(eligible_encounters))},
        {"metric": "reference_pairs_total", "value": int(len(anchored_pairs_df))},
        {"metric": "pairs_with_pulseox_match", "value": int(anchored_pairs_df['has_pulseox_match'].sum())},
        {"metric": "pairs_with_2hz_sample_match", "value": int(anchored_pairs_df['has_2hz_sample_match'].sum())},
        {"metric": "window_manifest_rows", "value": int(len(window_manifest_df))},
        {"metric": "window_manifest_positive_labels", "value": int(window_manifest_df['binary_label_so2_lt_90'].sum()) if len(window_manifest_df) else 0},
    ]
)

print(f"Wrote: {OPENOX_SAMPLE_REFERENCE_PAIRS_CSV}")
print(f"Wrote: {OPENOX_SAMPLE_WINDOW_MANIFEST_CSV}")
summary_df